# ReDrafter Training — Qwen2.5-7B on A100
Trains a small RNN draft head (Apple arxiv:2403.09919)
for token-level speculative decoding.

**Runtime:** GPU → A100 (Colab Pro)
**Expected training time:** ~2-3 hours (3000 steps)
**Expected speedup after deployment:** 1.3-1.5x on Apple Silicon

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes sentencepiece
print('✅ Dependencies installed')

In [ ]:
import os, gc, math, time, logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
BASE_MODEL        = 'Qwen/Qwen2.5-7B-Instruct'
DRAFTER_HIDDEN    = 512
DRAFTER_LAYERS    = 2
NUM_DRAFT_TOKENS  = 4
MAX_SEQ_LEN       = 512   # Proven safe on A100 40GB
MAX_SAMPLES       = 50_000
BATCH_SIZE        = 8
GRAD_ACCUM        = 4     # Effective batch = 32
LEARNING_RATE     = 2e-4
WARMUP_STEPS      = 200
MAX_STEPS         = 3000
SAVE_EVERY        = 500
LOG_EVERY         = 50
OUTPUT_DIR        = '/content/drafter_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Config set')

In [ ]:
# ── Load base model (frozen) ─────────────────────────────────────────────
print(f'Loading tokenizer: {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
vocab_size = len(tokenizer)
print(f'Vocab size: {vocab_size}')

print(f'Loading base model (bfloat16, CUDA)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad_(False)

hidden_size = base_model.config.hidden_size
print(f'✅ Base model loaded. Hidden size: {hidden_size}')

In [ ]:
# ── ReDrafter Head ───────────────────────────────────────────────────────
class ReDrafterHead(nn.Module):
    def __init__(self, vocab_size, hidden_size, drafter_hidden=512, num_layers=2, num_draft_tokens=4):
        super().__init__()
        self.num_draft_tokens = num_draft_tokens
        self.input_proj  = nn.Linear(hidden_size, drafter_hidden)
        self.gru = nn.GRU(drafter_hidden, drafter_hidden, num_layers=num_layers, batch_first=True)
        self.token_embed = nn.Embedding(vocab_size, drafter_hidden)
        self.output_proj = nn.Linear(drafter_hidden, vocab_size, bias=False)
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)
        nn.init.xavier_uniform_(self.output_proj.weight)

    def forward(self, hidden_state, target_ids=None):
        B = hidden_state.size(0)
        h0 = self.input_proj(hidden_state).unsqueeze(0).expand(
            self.gru.num_layers, -1, -1).contiguous()
        x = torch.zeros(B, h0.shape[-1], device=hidden_state.device, dtype=hidden_state.dtype)
        h, logits_list = h0, []
        for step in range(self.num_draft_tokens):
            out, h = self.gru(x.unsqueeze(1), h)
            logits = self.output_proj(out.squeeze(1))
            logits_list.append(logits)
            x = self.token_embed(target_ids[:, step] if target_ids is not None else logits.argmax(-1))
        return torch.stack(logits_list, dim=1)  # [B, K, V]

draft_head = ReDrafterHead(
    vocab_size, hidden_size, DRAFTER_HIDDEN, DRAFTER_LAYERS, NUM_DRAFT_TOKENS
).to(device).to(torch.bfloat16)
print(f'✅ Draft head: {sum(p.numel() for p in draft_head.parameters()):,} params')

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────
class ShareGPTDataset(Dataset):
    def __init__(self, tokenizer, max_seq=512, max_samples=50000):
        ds = load_dataset(
            'anon8231489123/ShareGPT_Vicuna_unfiltered',
            data_files='ShareGPT_V3_unfiltered_cleaned_split.json',
            split='train'
        )
        self.samples = []
        for row in ds.select(range(min(max_samples, len(ds)))):
            text = ' '.join(t.get('value','') for t in row.get('conversations',[]))
            if len(text) < 100: continue
            ids = tokenizer(text, truncation=True, max_length=max_seq+1, return_tensors='pt').input_ids.squeeze(0)
            if len(ids) >= 8: self.samples.append(ids)
        print(f'Dataset: {len(self.samples)} samples')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate(batch):
    ml = max(x.size(0) for x in batch)
    p = torch.zeros(len(batch), ml, dtype=torch.long)
    for i,x in enumerate(batch): p[i,:x.size(0)] = x
    return p

dataset = ShareGPTDataset(tokenizer, MAX_SEQ_LEN, MAX_SAMPLES)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=True)

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────
try:
    from bitsandbytes.optim import AdamW8bit
    optimizer = AdamW8bit(draft_head.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    print('Using 8-bit AdamW')
except Exception:
    from torch.optim import AdamW
    optimizer = AdamW(draft_head.parameters(), lr=LEARNING_RATE, weight_decay=0.01, betas=(0.9,0.95))
    print('Using standard AdamW')

def get_lr(step):
    if step < WARMUP_STEPS: return LEARNING_RATE * step / max(WARMUP_STEPS,1)
    p = (step-WARMUP_STEPS) / max(MAX_STEPS-WARMUP_STEPS,1)
    return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * p))

step = 0
optimizer.zero_grad()
t0 = time.time()
new_lr = LEARNING_RATE

print('Starting training...')
while step < MAX_STEPS:
    for batch in loader:
        if step >= MAX_STEPS: break
        batch = batch.to(device)
        sl = batch.size(1)
        if sl < NUM_DRAFT_TOKENS + 2: continue
        start = torch.randint(1, max(2, sl-NUM_DRAFT_TOKENS-1), (1,)).item()
        ctx = batch[:, :start]
        tgt = batch[:, start:start+NUM_DRAFT_TOKENS]

        with torch.no_grad():
            out = base_model(input_ids=ctx, output_hidden_states=True, use_cache=False)
            h = out.hidden_states[-1][:, -1, :]

        logits = draft_head(h, target_ids=tgt)
        # Memory-efficient loss
        loss = sum(F.cross_entropy(logits[:,k,:].float(), tgt[:,k], ignore_index=0)
                   for k in range(NUM_DRAFT_TOKENS)) / NUM_DRAFT_TOKENS
        (loss / GRAD_ACCUM).backward()

        if (step+1) % GRAD_ACCUM == 0:
            new_lr = get_lr(step)
            for pg in optimizer.param_groups: pg['lr'] = new_lr
            optimizer.step()
            optimizer.zero_grad()

        step += 1
        if step % LOG_EVERY == 0:
            print(f'step={step}/{MAX_STEPS} | loss={loss.item():.4f} | lr={new_lr:.2e} | {time.time()-t0:.0f}s')
        if step % SAVE_EVERY == 0:
            p = f'{OUTPUT_DIR}/redrafter_qwen25-7b_step{step}.pt'
            torch.save(draft_head.state_dict(), p)
            print(f'  Saved: {p}')
        gc.collect()

torch.save(draft_head.state_dict(), f'{OUTPUT_DIR}/redrafter_qwen25-7b_final.pt')
print(f'✅ Training complete in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Download weights ─────────────────────────────────────────────────────
import shutil
shutil.make_archive('/content/drafter_weights', 'gztar', OUTPUT_DIR)
from google.colab import files
files.download('/content/drafter_weights.tar.gz')
print('✅ Weights downloaded!')